In [50]:
from langchain_community.utilities import SQLDatabase
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
import pandas as pd

In [12]:
db = SQLDatabase.from_uri(
    "sqlite:///../data/Chinook.db"
)

In [ ]:
print(db.get_usable_table_names())

In [ ]:
table_info = db.get_table_info()

print(table_info)

In [ ]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

print(groq_api_key is not None)

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)

In [ ]:
response = llm.invoke("Say hello in one sentence.")

print(response.content)

In [17]:
question = "Which artist has the most tracks?"

prompt = f"""
You are an expert SQLite SQL developer.

Given the following database schema:

{table_info}

Generate a valid SQLite SQL query that answers the user's question.

User Question:
{question}

Rules:
- Generate only valid SQLite SQL.
- Use only the tables and columns provided in the schema.
- Return ONLY the SQL query.
- Do not include explanations.
- Do not use markdown code blocks.

SQL:
"""

In [ ]:
response = llm.invoke(prompt)

generated_sql = response.content

print(generated_sql)

In [ ]:
result = db.run(generated_sql)
print(result)

In [20]:
def generate_sql(question):
    
    prompt = f"""
    You are an expert SQLite SQL developer.

    Given the following database schema:

    {table_info}

    Generate a valid SQLite SQL query that answers the user's question.

    User Question:
    {question}

    Rules:
    - Generate only valid SQLite SQL.
    - Use only the tables and columns provided in the schema.
    - Return ONLY the SQL query.
    - Do not include explanations.
    - Do not use markdown code blocks.

    SQL:
    """

    response = llm.invoke(prompt)

    return response.content

In [ ]:
question = "Which country has the highest number of customers?"

generated_sql = generate_sql(question)

print(generated_sql)

In [ ]:
result = db.run(generated_sql)

print(result)

In [23]:
def execute_sql(sql_query):
    result = db.run(sql_query)
    return result

In [ ]:
question = "Which country has the highest number of customers?"

generated_sql = generate_sql(question)

print("Generated SQL:")
print(generated_sql)

result = execute_sql(generated_sql)

print("\nResult:")
print(result)

In [25]:
def ask_database(question):
    sql = generate_sql(question)
    result = execute_sql(sql)

    return sql, result

In [ ]:
question = "Show the top 5 customers by total spending"

sql, result = ask_database(question)

print("Generated SQL:")
print(sql)

print("\nResult:")
print(result)

In [31]:
def validate_sql(sql_query):
    sql_query = sql_query.strip()

    # Remove one semicolon at the end
    if sql_query.endswith(";"):
        sql_query = sql_query[:-1].strip()

    # Only allow SELECT queries
    if not sql_query.upper().startswith("SELECT"):
        return False

    # Block multiple statements
    if ";" in sql_query:
        return False

    return True

In [ ]:
print(validate_sql("SELECT * FROM Customer;"))

In [ ]:
print(validate_sql("SELECT * FROM Products; DELETE FROM Customer;"))

In [48]:
def execute_sql(sql_query):

    if not validate_sql(sql_query):
        return False, "Error: Only a single SELECT query is allowed."

    try:
        result = result = pd.read_sql_query(sql_query, db._engine)
        return True, result

    except Exception as e:
        return False, str(e)

In [ ]:
valid_sql = """
SELECT *
FROM Customer
LIMIT 3;
"""

print(execute_sql(valid_sql))

In [ ]:
dangerous_sql = """
SELECT * FROM Customer;
DELETE FROM Customer;
"""

print(execute_sql(dangerous_sql))

In [38]:
def ask_database(question):
    sql = generate_sql(question)
    result = execute_sql(sql)

    return sql, result

In [ ]:
question = "Show the top 5 customers by total spending"

sql, result = ask_database(question)

print("Generated SQL:")
print(sql)

print("\nResult:")
print(result)

In [40]:
def fix_sql(question, sql_query, error):

    prompt = f"""
You are an expert SQLite SQL developer.

The following SQL query produced an error.

Database schema:
{table_info}

User question:
{question}

Incorrect SQL:
{sql_query}

Database error:
{error}

Fix the SQL query.

Rules:
- Generate only valid SQLite SQL.
- Use only tables and columns from the schema.
- Return ONLY the corrected SQL.
- Do not include markdown or explanations.
"""

    response = llm.invoke(prompt)

    return response.content

In [54]:
def ask_database(question):

    # Step 1: Generate SQL
    sql = generate_sql(question)

    # Step 2: Execute SQL
    success, result = execute_sql(sql)

    # Step 3: Fix SQL if execution fails
    if not success:
        print("Initial SQL failed. Trying to fix it...")

        sql = fix_sql(
            question=question,
            sql_query=sql,
            error=result
        )

        success, result = execute_sql(sql)

    # Step 4: Generate natural language answer
    if success:
        answer = generate_answer(question, result)

        return {
            "question": question,
            "sql": sql,
            "result": result,
            "answer": answer
        }

    # If SQL still fails
    return {
        "question": question,
        "sql": sql,
        "result": None,
        "answer": f"Unable to execute the query: {result}"
    }

In [ ]:
response = ask_database(
    "Show the top 5 customers by total spending"
)

print("Question:")
print(response["question"])

print("\nGenerated SQL:")
print(response["sql"])

print("\nResult:")
print(response["result"])

In [ ]:
response = ask_database(
    "Show the top 5 customers by total spending"
)

print("Generated SQL:")
print(response["sql"])

print("\nResult:")
display(response["result"])

In [52]:
def generate_answer(question, result):
    
    prompt = f"""
You are a helpful data assistant.

Answer the user's question using only the SQL query result provided.

User Question:
{question}

SQL Result:
{result.to_string(index=False)}

Rules:
- Give a clear and concise answer.
- Use only information present in the SQL result.
- Do not make up information.
- Do not mention SQL unless necessary.
"""

    response = llm.invoke(prompt)

    return response.content

In [ ]:
response = ask_database(
    "Show the top 5 customers by total spending"
)
print("QUESTION:")
print(response["question"])

print("\nGENERATED SQL:")
print(response["sql"])

print("\nRESULT:")
display(response["result"])

print("\nANSWER:")
print(response["answer"])

In [ ]:
questions = [
    "How many customers are there?",
    "Show all customers from Germany.",
    "Which country has the highest number of customers?",
    "Show the top 5 most expensive tracks.",
    "List all albums created by AC/DC.",
    "Which artist has the most tracks?",
    "Show the top 5 customers by total spending.",
    "Which music genre has the highest number of tracks?",
    "Which artist generated the highest revenue from track sales?",
    "Show the top 5 countries by total revenue."
]

for i, question in enumerate(questions, start=1):
    
    print("=" * 70)
    print(f"QUESTION {i}: {question}")
    print("=" * 70)

    response = ask_database(question)

    print("\nGENERATED SQL:")
    print(response["sql"])

    print("\nRESULT:")
    
    if response["result"] is not None:
        display(response["result"])

    print("\nANSWER:")
    print(response["answer"])

    print("\n")